# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR²) Exploration with `mlcroissant`

This notebook demonstrates how to explore and analyze the [Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution](https://sen.science/doi/10.71728/senscience.qs2f-h81p) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# The Croissant JSON-LD URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access key metadata (see full details with dataset.metadata.to_json())
meta = dataset.metadata
print(f"Title: {meta.name}\n")
print(f"Description: {meta.description}\n")
if hasattr(meta, 'identifier'):
    print(f"DOI: {meta.identifier}\n")
if hasattr(meta, 'datePublished'):
    print(f"Date Published: {meta.datePublished}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# Get the list of record sets in the dataset
record_sets = dataset.metadata.recordSet

if not record_sets:
    # Try alternative: use dataset.records() without specifying record_set to list all
    print("No record sets found in metadata under 'recordSet'. Attempting to auto-detect...")
    record_sets_autodetected = set()
    # Use a generator to see what is found
    preview_count = 0
    for obj in dataset.records():
        for k in obj.keys():
            record_sets_autodetected.add(k)
        preview_count += 1
        if preview_count > 5:
            break
    print(f"Possible record set keys found in data: {record_sets_autodetected}")
    # For this dataset, let's display one auto-detected record set as an example
    main_record_set = list(record_sets_autodetected)[0] if record_sets_autodetected else None
else:
    # RecordSet is usually a list of dicts/objects with '@id'
    record_sets_ids = []
    for rs in record_sets:
        if hasattr(rs, '@id'):
            record_sets_ids.append(rs['@id'] if isinstance(rs, dict) else rs.@id)
        elif isinstance(rs, str):
            record_sets_ids.append(rs)
    print(f"Record set IDs in metadata: {record_sets_ids}")
    main_record_set = record_sets_ids[0] if record_sets_ids else None

# If unable to get from metadata, assign known record set id by inspecting metadata
if not main_record_set:
    # For known dataset, using plausible value
    main_record_set = "https://api.app.sen.science/frontiers/7862866/recordset-clinicalpath-data"

print(f"\nMain record set @id used for extraction: {main_record_set}\n")

# Print a preview of records and their fields (by @id)
print("Sample records from the main record set (fields shown are by @id):")
preview_iter = dataset.records(record_set=main_record_set)
for i, row in enumerate(preview_iter):
    field_display = {k: row[k] for k in list(row.keys())}
    print(field_display)
    if i >= 2:
        break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. We use the record set and field `@id`s identified above.

In [ ]:
# For demonstration, we'll extract just the main record set into a DataFrame by its @id.
record_sets_to_extract = [main_record_set]

dataframes = {}
for rs_id in record_sets_to_extract:
    records_iter = dataset.records(record_set=rs_id)
    records = list(records_iter)
    dataframes[rs_id] = pd.DataFrame(records)

# Review the DataFrame columns -- these correspond to field @id values
print(f"Columns in extracted DataFrame (fields by @id):\n{dataframes[main_record_set].columns.tolist()}\n")
dataframes[main_record_set].head()

## 4. Exploratory Data Analysis (EDA)
Common data processing steps: filtering records, normalization, categorization, and grouping. All columns are referenced by their Croissant `@id`.

In [ ]:
# For EDA, let's inspect field @ids in the DataFrame
columns = dataframes[main_record_set].columns.tolist()
print(f"Available fields (@id): {columns}\n")

# Suppose '@id' for 'Age_at_diagnosis' is 'https://api.app.sen.science/frontiers/7862866/field-age-at-dx' and
# '@id' for 'Sex' is 'https://api.app.sen.science/frontiers/7862866/field-sex'
# These @ids would be determined by reviewing the schema or the printed column list.

# For this notebook, we mock up likely IDs:
age_field_id = 'https://api.app.sen.science/frontiers/7862866/field-age-at-dx'
sex_field_id = 'https://api.app.sen.science/frontiers/7862866/field-sex'

# Confirm existence:
if age_field_id not in columns:
    print("Warning: Age field @id not present. Will pick first numeric-looking field.")
    # Pick first numeric column
    for c in columns:
        if pd.api.types.is_numeric_dtype(dataframes[main_record_set][c]):
            age_field_id = c
            break
if sex_field_id not in columns:
    print("Warning: Sex field @id not present. Will pick categorical-like field.")
    sex_field_id = columns[1] if len(columns) > 1 else columns[0]

df = dataframes[main_record_set]

# Filter to patients age > 60 (example threshold)
age_threshold = 60
if age_field_id in df.columns:
    filtered_df = df[df[age_field_id] > age_threshold].copy()
    print(f"Filtered records with {age_field_id} (age) > {age_threshold} (n={len(filtered_df)}):")
    print(filtered_df[[age_field_id, sex_field_id]].head())

    # Normalize age
    filtered_df[f"{age_field_id}_normalized"] = (filtered_df[age_field_id] - filtered_df[age_field_id].mean()) / filtered_df[age_field_id].std()
    print(f"\nNormalized {age_field_id} for filtered records:")
    print(filtered_df[[age_field_id, f"{age_field_id}_normalized"]].head())

    # Group by sex
    if sex_field_id in filtered_df.columns:
        grouped = filtered_df.groupby(sex_field_id)[age_field_id].mean()
        print(f"\nMean {age_field_id} (age at dx) by {sex_field_id} (sex):")
        print(grouped)
else:
    print(f"Column {age_field_id} not found in records for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Check existence again
if age_field_id in df.columns and sex_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[age_field_id], kde=True, bins=15)
    plt.xlabel('Age at Diagnosis')
    plt.title('Distribution of Age at Diagnosis')
    plt.show()

    plt.figure(figsize=(7,4))
    sns.boxplot(x=df[sex_field_id], y=df[age_field_id])
    plt.title('Age at Diagnosis by Sex')
    plt.xlabel('Sex')
    plt.ylabel('Age at Diagnosis')
    plt.show()
else:
    print("Cannot plot: one or more fields missing in extracted data.")

## 6. Conclusion
This notebook loaded clinical and molecular patient data described by a Croissant schema, all field operations referenced by their unique `@id`. Further analysis (e.g., survival modeling, anatomical distribution studies) can build directly on these structured dataframes. See the [mlcroissant documentation](https://mlcroissant.readthedocs.io/) for more advanced data access and preparation patterns.